# NB2 · Preparing the data

**Building Clinical Decision Support Systems with Generative AI Tools**  
Technology and Artificial Intelligence Literacy Training in Health Sciences · Akdeniz University · 18 September 2026

Prof. Dr. Utku Köse · Süleyman Demirel University, Department of Computer Engineering  
Director, Artificial Intelligence Application and Research Center (YAZEM) · utkukose@sdu.edu.tr

---


## What this notebook does

Raw data cannot be handed to a model. Three things are needed: cleaning, the train and
test split, and conversion into the form the model can use.

The order matters. Conversion comes after the split; you will see why in step three.


## Setup


In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib

import sys, urllib.request

REPO = 'https://raw.githubusercontent.com/utkukose/cdss-genai-NB-lecture/main'
urllib.request.urlretrieve(f'{REPO}/workshop/cdss_kit.py?v={id(object())}', 'cdss_kit.py')

sys.modules.pop('cdss_kit', None)   # drop any stale copy from the session
import cdss_kit as kit
kit.LANG = 'en'

print('Ready. Helper version:', kit.SURUM)


---

## Code carried from the previous notebook

Paste the whole block from the end of the previous notebook into the cell below and run
it. Do not delete the marker on the first line.


In [ ]:
#@cdss onceki_defter
# Paste the generated code below this line.


---

## Step 1 · Cleaning

Real clinical data does not arrive complete. Some measurements were never taken, some were
entered wrongly, some records are duplicated.


### Prompt

```
Write a function that cleans the cohort dataframe. Name it clean, let it clean whatever
frame it is given and return the cleaned version.

Drop entirely empty columns. Drop duplicate rows. Drop rows where target is empty. Treat
physiologically impossible values as missing; do not delete the row, empty only that cell.
State in a comment which bound you used for which variable.

Print how many columns and rows were dropped. Run it on cohort and keep the result in
clean_data.
```


In [ ]:
#@cdss temizleme
# Paste the generated code below this line.


### Look at your code

- Which bound was used for which variable? Who should set those bounds, you or the tool?
- Was the row deleted where an impossible value was found, or only that cell emptied?
- How many rows remain? Did the event rate change after cleaning?


In [ ]:
print('Before cleaning:', len(cohort), 'rows')
print('After cleaning :', len(clean_data), 'rows')
print('Event rate     :', f"{clean_data['target'].mean():.1%}")


---

## Step 2 · Train and test split

A model learns from the data it is given. Tested on the same data, it measures what it
memorised. The data is therefore split in two.

How the split is made is critical. In NB1 you saw that the patient count can be lower than
the row count. Split rows at random and one record of a patient falls into training and
another into test; the model recognises that patient and the test result comes out better
than it is. **The split must be made at patient level.** AI tools do not do this unasked.


### Prompt

```
Split the clean_data dataframe into training and test sets.

Make the split at patient level: all records of a patient stay in one group, no patient
appears in both. The identifier is in patient_id. Use a test fraction of 0.30 and
RANDOM_SEED.

Keep the results in train and test. Print the row count, patient count and event rate of
each.
```


In [ ]:
#@cdss ayrim
# Paste the generated code below this line.


### Look at your code


In [ ]:
shared = set(train['patient_id']) & set(test['patient_id'])
print('Patients in both groups:', len(shared))
print('Train:', len(train), 'rows,', train['patient_id'].nunique(), 'patients')
print('Test :', len(test), 'rows,', test['patient_id'].nunique(), 'patients')
print('Event rate — train:', f"{train['target'].mean():.1%}",
      '| test:', f"{test['target'].mean():.1%}")


- Is the shared patient count zero? If not, the split was made at row level; remind the
  tool and regenerate.
- Are the two event rates close? If they are far apart, you are seeing how much a single
  split can shift results on a small dataset.
- Which line performs the split? Can you find where the patient identifier is used?


---

## Step 3 · Preparing for the model

A model works with numbers. Categorical variables have to be encoded, missing values
filled, numeric variables scaled.

**These conversions are learned from the training set alone.** If a median is used to fill
missing values, it is computed on the training set and applied unchanged to the test set.
Computed across all the data, the test set leaks into training.

That is why step three follows the split.


### Prompt

```
Convert the train and test frames into the form the model can use.

Separate the target column into y_train and y_test. Do not give patient_id or other
identifier columns to the model.

Apply the conversion appropriate to DATASET:
  tabular : encode categorical variables, fill missing values, scale numeric variables
  image   : bring images to the same size, scale pixel values, flatten
  signal  : derive summary features from each recording and say which you chose
  text    : convert the text to a numeric representation, build the vocabulary from the
            training set only

The conversions must be learned from the training set alone and applied unchanged to the
test set. Explain in a comment how you ensured this.

Keep the results in X_train, X_test, y_train, y_test and print their shapes.
```


In [ ]:
#@cdss hazirlama
# Paste the generated code below this line.


### Look at your code

- Where are the `fit` calls? Is `fit` called on the test set? If so there is leakage;
  have it corrected.
- How were missing values filled, and from which set was the filling value computed?
- How were categorical variables encoded? What happens if the test set holds a category
  never seen in training?


In [ ]:
import numpy as np
print('X_train:', np.shape(X_train), '| y_train:', np.shape(y_train))
print('X_test :', np.shape(X_test), '| y_test :', np.shape(y_test))
print('Same column count:', np.shape(X_train)[1] == np.shape(X_test)[1])


---

## End of notebook · Collect the code

The cell below returns what you wrote here as one block. Copy it into the first cell of
NB3. The cell also downloads the file to your computer; if the download does not start,
take it from the file panel on the left.


In [ ]:
code_so_far = kit.export('cdss_nb2.py')


## What this notebook did

The data was cleaned, split at patient level and prepared for the model. Two routes for
leakage were closed: the patient level split, and learning the conversions from the
training set alone. Both raise no error, improve the result and are therefore hard to
notice.

NB3 builds a model and measures it.
---

**Warning.** Nothing produced in this notebook is a validated clinical tool. The dataset
you used is an open collection prepared for teaching and does not represent the patient
population of your own institution.
